# Product Query Agent

In [6]:
from dotenv import load_dotenv
# from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent

load_dotenv()

True

In [12]:
from typing import Annotated

PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: Annotated[str, "Product name, e.g. 'mechanical keyboard'"]) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

In [17]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)


agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt=(
        "You are a helpful product assistant for an online tech store. "
        "Always call a tool to look up a product before answering. Never guess prices or details."
    ),
)

In [18]:
def ask(question: str):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [19]:
ask("what is the price of wireless headphones.")

The wireless headphones are priced at **$79.99**.

Here are a few more details about them:
*   **Rating:** 4.6 stars
*   **Features:** Over-ear Bluetooth design, 30-hour battery life, and active noise cancellation.


In [ ]:
ask("tell me about mechanical keyboard")

In [21]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool
def get_review(name: Annotated[str, "Product name, e.g. 'smart watch'"]) -> str:
    """Look up a product review by a product name. Return the product name, number of reviews and rating"""
    r = REVIEWS.get(name.lower())
    if not r:
        return f"Review not available for this product"
    return str(r)

In [26]:
# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)


agent2 = create_agent(
    llm,
    tools=[get_product, get_review],
    system_prompt=(
        "You are a helpful product assistant for an online tech store. "
        "Always call a tool to look up a product before answering. Never guess prices or details."
    ),
)

def ask2(question: str):
    result = agent2.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [27]:
ask2("how do people like smart watch")

People generally like the smart watch! It has an average rating of **3.9 out of 5** based on **340 reviews**.

This suggests that while it’s a well-received product, there’s still room for improvement—some customers may have had mixed experiences. If you’re considering buying one, I’d recommend checking out the detailed reviews to see what specific features people love or dislike. Would you like help looking up more details about the smart watch, such as its price, stock availability, or description?


In [28]:
ask2("what is the price and reviews of smart watch")

Here are the details for the **Smart Watch**:

*   **Price:** $199.99
*   **Reviews:** 340 reviews with an average rating of **3.9/5**
*   **Product Rating:** 4.3/5
*   **Description:** Tracks heart rate and sleep. Features a 5-day battery and is water-resistant.
